In [ ]:
import librosa
import matplotlib.pyplot as plt
import soundfile as sf

import torch
from IPython.display import Audio
import random
from safetensors.torch import load_file
from models.demucs_equalizer import DemucsEqualizer, DoubleDemucsEqualizer

In [ ]:
import numpy as np
import librosa
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import pearsonr

def mse_time_domain(reference, estimated):
    # import numpy as np
    reference = (reference - reference.min()) / (reference.max() - reference.min())
    estimated = (estimated - estimated.min()) / (estimated.max() - estimated.min())
    return np.mean((reference - estimated) ** 2)

def mse_freq_domain(reference, estimated):
    # import numpy as np
    delta = 1e-7 
    reference_db = librosa.amplitude_to_db(np.abs(librosa.stft(reference)))
    estimated_db = librosa.amplitude_to_db(np.abs(librosa.stft(estimated)))
    reference_db = (reference_db - reference_db.min()) / (reference_db.max() - reference_db.min())
    estimated_db = (estimated_db - estimated_db.min()) / (estimated_db.max() - estimated_db.min())
    return np.mean((reference_db - estimated_db) ** 2)

def sdr(references, estimates):
    # import numpy as np
    delta = 1e-7 
    num = np.sum(np.square(references))
    den = np.sum(np.square(references - estimates))
    num += delta
    den += delta
    return 10 * np.log10(num / den)

def sisnr(reference, estimated):
    # import numpy as np
    reference = reference - np.mean(reference)
    estimated = estimated - np.mean(estimated)
    reference_energy = np.sum(reference ** 2)
    projection = np.sum(reference * estimated) * reference / reference_energy
    noise = estimated - projection
    si_snr = 10 * np.log10(np.sum(projection ** 2) / np.sum(noise ** 2))
    return si_snr

def calculate_average(reference_list, estimated_list,function):
    metrix_values = [function(reference, estimated) for reference, estimated in zip(reference_list, estimated_list)]
    average_value = np.mean(metrix_values)
    return average_value

In [4]:
def split_audio(audio,sr=44100,total_len=300,segment_len=5):
    total_samples = total_len*sr
    segment_samples = segment_len*sr
    segments_list = [audio[i:i + segment_samples] for i in range(0, total_samples, segment_samples)]
    return segments_list

def join_audio(segments: list):
    import numpy as np
    return np.concatenate(segments, axis=0)

In [5]:
def get_prediction_from_audio(seq,x_path, y1_path, y2_path):
    device = 'cpu'
    x,sr = librosa.load(x_path+f'{seq}.wav',sr=None)
    y1,sr = librosa.load(y1_path+f'{seq}.wav',sr=None)
    y2,sr = librosa.load(y2_path+f'{seq}.wav',sr=None)
    x_list = split_audio(x)
    y1_list = split_audio(y1)
    y2_list = split_audio(y2)
    y1_predict_list,gx_list,y2_predict_list = [],[],[]
    
    # change your model here
    checkpoint_path = "assets/v2/60_stage1/pytorch_model.bin"
    model = DemucsEqualizer(device=device)
    state_dict = torch.load(checkpoint_path, map_location=device)
    state_dict = {k[6:]: v for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    model.eval()  
    model = model.to(device)
    for i in range(len(x_list)):
        with torch.no_grad():
            y1_predict = model(torch.tensor(x_list[i]).unsqueeze(0).unsqueeze(0)).squeeze(0).squeeze(0).cpu().numpy()
            y1_predict_list.append(y1_predict)

    # change your model here
    checkpoint_path = "assets/v2/60_stage2/pytorch_model.bin"
    model = DoubleDemucsEqualizer("assets/v2/60_stage1/pytorch_model.bin", device=device)
    state_dict = torch.load(checkpoint_path, map_location=device)
    state_dict = {k[6:]: v for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    model.eval()
    model = model.to(device)
    for i in range(len(x_list)):
        input_waveform = torch.tensor(x_list[i]).unsqueeze(0).unsqueeze(0)
        if len(torch.tensor(x_list[seq]).shape) == 2:
            input_waveform = input_waveform.unsqueeze(1)
            input_waveform = torch.cat([input_waveform, input_waveform], dim=1)       
        input_waveform = torch.cat([input_waveform, input_waveform], dim=1)        
        with torch.no_grad():
            first_waveform = model.model2(input_waveform)
            first_waveform = first_waveform.mean(dim=1)[:,0,:]  
            first_waveform = first_waveform.reshape(first_waveform.shape[0], -1)
            gx_list.append(first_waveform.squeeze(0).cpu().numpy())
            second_waveform = model.model1(first_waveform).squeeze(0).cpu().numpy()  
            y2_predict_list.append(second_waveform)
    return x_list,y1_list,y1_predict_list,y2_list,gx_list,y2_predict_list


In [ ]:
from tqdm import tqdm 

total_stage1_results = {'mse_time_domain': 0, 'mse_freq_domain': 0, 'sdr': 0, 'sisnr': 0}
total_stage2_results = {'mse_time_domain': 0, 'mse_freq_domain': 0, 'sdr': 0, 'sisnr': 0}

for i in tqdm(range(1, 6)):
    x_list, y1_list, y1_predict_list, y2_list, gx_list, y2_predict_list = get_prediction_from_audio(
        i,
        x_path='data/v2/EN_x/',
        y1_path='data/v2/EN_y1/',
        y2_path='data/v2/EN_y2_headphone/'
    )
    for function in mse_time_domain, mse_freq_domain, sdr, sisnr:
        stage1_result = calculate_average(y1_list, y1_predict_list, function)
        stage2_result = calculate_average(y2_list, y2_predict_list, function)
        total_stage1_results[function.__name__] += stage1_result
        total_stage2_results[function.__name__] += stage2_result
average_stage1_results = {k: v / 60 for k, v in total_stage1_results.items()}
average_stage2_results = {k: v / 60 for k, v in total_stage2_results.items()}
print("Average results for Stage 1:", average_stage1_results)
print("Average results for Stage 2:", average_stage2_results)
# record the results
